# 01 · Preparación de los datos: problema de BCS

Lee de `data/` las fotos (`fotosextraidas/`), la bitácora (`etiquetas_bsc.xlsx`) y las vistas marcadas (`intermedios/vistas.csv`), y escribe el dataset en `data/dataset_bcs/`. Las fotos y la bitácora las descarga `01_du_bcs_problem.ipynb`. Los archivos originales no se modifican.

## Configuración

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import datetime as dt
import hashlib
import os
import re
import shutil

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from PIL import Image, ImageOps

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    LEE_HEIC = True
except ImportError:
    LEE_HEIC = False


def buscar_data():
    """Sube desde la carpeta del notebook hasta encontrar la carpeta data/ del repositorio."""
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "data").is_dir():
            return (carpeta / "data").resolve()
    raise FileNotFoundError("No encontré la carpeta data/. Revisa el README.")


DATA = buscar_data()
DIR_FOTOS = DATA / "fotosextraidas"
BITACORA = DATA / "etiquetas_bsc.xlsx"
INTER = DATA / "intermedios"
DIR_JPG = INTER / "fotos_jpg"
VISTAS = INTER / "vistas.csv"
SALIDA = DATA / "dataset_bcs"

EXCLUIR_IDS = {"2123"}
K = 5
CALIDAD = 95
HILOS = 4

INTER.mkdir(exist_ok=True)
pd.set_option("display.max_rows", 120)

ref_indice = SALIDA / "indice_imagenes.csv"
referencia = pd.read_csv(ref_indice, dtype=str, keep_default_na=False) if ref_indice.is_file() else None

print("Datos en:", DATA)
for p in (DIR_FOTOS, BITACORA):
    print("OK       " if p.exists() else "NO EXISTE", p.name)
print("Lectura de HEIC:", "sí" if LEE_HEIC else "no (instala pillow-heif)")
print("Índice de referencia:", f"sí ({len(referencia)} filas)" if referencia is not None else "no")

### Funciones auxiliares

In [ ]:
def sha256(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(bloque), b""):
            h.update(b)
    return h.hexdigest()


def es_copia_numerada(nombre):
    return bool(re.search(r"\(\d+\)$", Path(nombre).stem.strip()))


def nombre_base(nombre):
    return re.sub(r"\s*\(\d+\)$", "", Path(nombre).stem.strip())


def norm(t):
    t = str(t).lower().translate(str.maketrans("áéíóúüñ", "aeiouun"))
    return re.sub(r"[^a-z0-9]", "", t)


def texto(v):
    if v is None:
        return ""
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    if isinstance(v, (dt.time, dt.datetime)):
        return v.strftime("%H %M")
    return str(v).strip()


def leer_bitacora(ruta):
    wb = load_workbook(ruta, read_only=True, data_only=True)
    for ws in wb.worksheets:
        for i, fila in enumerate(ws.iter_rows(min_row=1, max_row=40, values_only=True), start=1):
            n = [norm(v) for v in fila if v not in (None, "")]
            if any(x.startswith("id") for x in n) and any(x.startswith("bcs") for x in n):
                filas = list(ws.iter_rows(min_row=i, values_only=True))
                wb.close()
                enc = [texto(c) or f"(columna {j + 1})" for j, c in enumerate(filas[0])]
                datos = []
                for num, f in enumerate(filas[1:], start=i + 1):
                    vals = [texto(c) for c in f][:len(enc)]
                    if any(vals):
                        datos.append([num] + vals + [""] * (len(enc) - len(vals)))
                return pd.DataFrame(datos, columns=["fila_excel"] + enc)
    wb.close()
    raise ValueError("No encontré la fila de encabezados con 'ID' y 'BCS'")


def columna(df, *prefijos):
    for c in df.columns:
        if c != "fila_excel" and any(norm(c).startswith(p) for p in prefijos):
            return c
    raise KeyError(prefijos)


def leer(ruta):
    df = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    df.columns = [str(c).replace("﻿", "").strip() for c in df.columns]
    return df


def clave(x):
    return (0, int(x)) if str(x).isdigit() else (1, str(x))

## 3.1 Seleccionar los datos

In [ ]:
EXT_FOTO = {".heic", ".heif", ".jpg", ".jpeg", ".png"}
filas = []
for p in sorted(DIR_FOTOS.rglob("*")):
    if p.is_file() and p.suffix.lower() in EXT_FOTO and not p.name.startswith("."):
        rel = p.relative_to(DIR_FOTOS)
        filas.append({"id_vaca": p.parent.name.strip(), "ruta": str(rel),
                      "descarga": rel.parts[0] if len(rel.parts) > 1 else "",
                      "nombre": p.name, "ext": p.suffix.lower(), "bytes": p.stat().st_size,
                      "sha256": sha256(p), "copia_numerada": es_copia_numerada(p.name)})
todas = pd.DataFrame(filas)

bit_raw = leer_bitacora(BITACORA)
c_id, c_hora, c_bcs = columna(bit_raw, "idvaca", "id"), columna(bit_raw, "hora"), columna(bit_raw, "bcs")
print(f"Fotos leídas: {len(todas)} | Filas de la bitácora: {len(bit_raw)}")
print(f"Columnas seleccionadas de la bitácora: '{c_id}', '{c_hora}', '{c_bcs}'")

**Resultados:** Se leyeron 477 archivos de fotos y 83 filas de la bitácora.

## 3.2 Limpiar los datos

### Copias exactas

In [ ]:
todas = todas.sort_values(["id_vaca", "sha256", "copia_numerada", "descarga", "nombre"])
copias = todas.groupby(["id_vaca", "sha256"]).size().rename("copias")
unicas = (todas.drop_duplicates(["id_vaca", "sha256"], keep="first")
               .merge(copias, on=["id_vaca", "sha256"]).reset_index(drop=True))
print(f"Archivos: {len(todas)} | copias quitadas: {len(todas) - len(unicas)} | fotos únicas: {len(unicas)}")

**Resultados:** Se quitaron 122 copias exactas y quedaron 355 fotos únicas.

### Exclusiones

In [ ]:
vacas_por_foto = unicas.groupby("sha256")["id_vaca"].apply(lambda s: sorted(set(s)))
compartidas = vacas_por_foto[vacas_por_foto.map(len) > 1]


def motivos(fila):
    m = []
    if fila["id_vaca"] in EXCLUIR_IDS:
        m.append("vaca excluida por decision del equipo")
    if fila["sha256"] in compartidas.index:
        m.append("misma foto en varias vacas: " + ", ".join(compartidas[fila["sha256"]]))
    return "; ".join(m)


unicas["motivo"] = unicas.apply(motivos, axis=1)
unicas["excluida"] = np.where(unicas["motivo"] != "", "si", "no")
display(unicas.loc[unicas["excluida"] == "si", "motivo"].value_counts().to_frame("fotos"))
print(f"Fotos que siguen: {(unicas['excluida'] == 'no').sum()} de {unicas.loc[unicas['excluida'] == 'no', 'id_vaca'].nunique()} vacas")

**Resultados:** Se excluyeron 22 fotos: 14 de la vaca 2123 y 8 filas repetidas entre la 1548 y la 2107 (las mismas 4 fotos en cada vaca). Siguen 333 fotos de 67 vacas.

### Bitácora

In [ ]:
bit = pd.DataFrame({"fila_excel": bit_raw["fila_excel"]})
bit["id_vaca"] = bit_raw[c_id].str.strip().str.replace(r"\.0$", "", regex=True)
h = bit_raw[c_hora].astype(str).str.strip()
m = h.str.extract(r"^(\d{1,2})\s+(\d{2})$")
bit["hora"] = (m[0].str.zfill(2) + ":" + m[1]).where(m[0].notna(), "")
bit["hora_original"] = h
bit["bcs"] = pd.to_numeric(bit_raw[c_bcs].astype(str).str.replace(",", ".").str.strip(), errors="coerce")
bit["excluida"] = np.where(bit["id_vaca"].isin(EXCLUIR_IDS), "si", "no")
bit["motivo"] = np.where(bit["excluida"] == "si", "vaca excluida por decision del equipo", "")

b = bit["bcs"]
print(f"BCS no numérico: {b.isna().sum()} | fuera de 1 a 5: {((b < 1) | (b > 5)).sum()} | "
      f"no múltiplo de 0.25: {(b.notna() & ((b * 4) % 1 != 0)).sum()}")
print(f"Evaluaciones excluidas: {(bit['excluida'] == 'si').sum()}")
bit.to_csv(INTER / "bitacora.csv", index=False)

**Resultados:** La bitácora limpia tiene 81 evaluaciones; se excluyeron 2. No hubo BCS no numéricos ni fuera de rango.

## 3.5 Formatear los datos (primera parte): conversión a JPG

Va antes de 3.3 porque la página para marcar las vistas no puede mostrar fotos HEIC.

In [ ]:
def procesar_foto(origen, destino, convertir):
    origen, ext = Path(origen), Path(origen).suffix.lower()
    r = {"ancho": None, "alto": None, "fecha_hora": "", "camara": "", "profundidad": "", "estado": "", "error": ""}
    try:
        with Image.open(origen) as im:
            exif = im.getexif()
            r["camara"] = f"{exif.get(0x010F, '')} {exif.get(0x0110, '')}".strip()
            fecha = str(exif.get_ifd(0x8769).get(0x9003, "") or exif.get(0x0132, "")).strip()
            r["fecha_hora"] = fecha.replace(":", "-", 2) if re.match(r"^\d{4}:\d{2}:\d{2}", fecha) else fecha
            if ext in {".heic", ".heif"}:
                r["profundidad"] = "si" if im.info.get("depth_images") else "no"
                girar = False
            else:
                r["profundidad"] = "no"
                girar = exif.get(0x0112, 1) not in (None, 1)
            final = ImageOps.exif_transpose(im) if girar else im
            r["ancho"], r["alto"] = final.size
            if not convertir:
                return {**r, "estado": "excluida"}
            destino = Path(destino)
            if destino.exists():
                return {**r, "estado": "ya existia"}
            destino.parent.mkdir(parents=True, exist_ok=True)
            tmp = destino.with_suffix(".tmp")
            if ext in {".jpg", ".jpeg"} and not girar:
                shutil.copy2(origen, tmp)
                r["estado"] = "copiada"
            else:
                exif[0x0112] = 1
                extra = {"quality": CALIDAD, "exif": exif.tobytes()}
                if im.info.get("icc_profile"):
                    extra["icc_profile"] = im.info["icc_profile"]
                final.convert("RGB").save(tmp, "JPEG", **extra)
                r["estado"] = "convertida"
            os.replace(tmp, destino)
    except Exception as e:
        r["estado"], r["error"] = "error", str(e).replace(str(origen), origen.name)[:200]
    return r


destinos, usados = [], set()
for _, f in unicas.iterrows():
    base = nombre_base(f["nombre"])
    if (f["id_vaca"], base.lower()) in usados:
        base = f"{base}_{f['sha256'][:8]}"
    usados.add((f["id_vaca"], base.lower()))
    destinos.append(f"fotos_jpg/{f['id_vaca']}/{base}.jpg")
unicas["archivo_jpg"] = np.where(unicas["excluida"] == "no", destinos, "")

tareas = [(DIR_FOTOS / f["ruta"], INTER / f["archivo_jpg"] if f["archivo_jpg"] else "", f["excluida"] == "no")
          for _, f in unicas.iterrows()]
with ThreadPoolExecutor(max_workers=HILOS) as ex:
    meta = pd.DataFrame(list(ex.map(lambda t: procesar_foto(*t), tareas)))
for c in ("ancho", "alto"):
    meta[c] = pd.to_numeric(meta[c], errors="coerce").astype("Int64")
fotos = pd.concat([unicas, meta], axis=1)
fotos.loc[fotos["estado"] == "error", "archivo_jpg"] = ""

fotos = (fotos.rename(columns={"ruta": "archivo_original", "ext": "formato_original", "bytes": "bytes_original"})
         [["id_vaca", "archivo_jpg", "excluida", "motivo", "archivo_original", "copias", "formato_original",
           "bytes_original", "sha256", "fecha_hora", "camara", "ancho", "alto", "profundidad", "estado", "error"]]
         .sort_values(["id_vaca", "archivo_original"]).reset_index(drop=True))
fotos.to_csv(INTER / "fotos.csv", index=False)

display(fotos["estado"].value_counts().to_frame("fotos"))
listas = fotos[(fotos["excluida"] == "no") & (fotos["archivo_jpg"] != "")]
tam = sum((INTER / r).stat().st_size for r in listas["archivo_jpg"] if (INTER / r).exists())
print(f"JPG listos: {len(listas)} ({tam / 1e9:.2f} GB) | con error: {(fotos['estado'] == 'error').sum()}")

**Resultados:** Se convirtieron 329 fotos y se copiaron sin cambios 4; quedaron 333 JPG ([COMPLETAR] GB). No hubo errores.

## Pausa: marcar la vista de cada foto

Las vistas se marcan a mano con `marcar_vistas.html` sobre la carpeta `data/intermedios/`: **1** posterior, **2** lateral izquierda, **3** lateral derecha y **0** descartar. El `vistas.csv` que genera se guarda en `data/intermedios/`. Si ya existe, el notebook sigue.

In [ ]:
assert VISTAS.is_file(), (
    f"Falta {VISTAS}. Marca las vistas con marcar_vistas.html y guarda ahí el vistas.csv.")
print("vistas.csv encontrado:", VISTAS)

## 3.3 Construir los datos

### Vista y si la foto es usable

In [ ]:
fotos_i = leer(INTER / "fotos.csv")
bit_i = leer(INTER / "bitacora.csv")
vis = leer(VISTAS)

f = fotos_i[(fotos_i["excluida"] == "no") & (fotos_i["archivo_jpg"] != "")].copy()
fv = f.merge(vis[["archivo_jpg", "vista", "usable", "marcado_por", "marcado_en"]], on="archivo_jpg", how="left")
for c in ("vista", "usable", "marcado_por", "marcado_en"):
    fv[c] = fv[c].fillna("")
fv["carpeta"] = np.where(fv["usable"] == "si", fv["vista"],
                         np.where(fv["usable"] == "no", "descartadas", "sin_marcar"))
fv["ruta"] = "imagenes/" + fv["carpeta"] + "/" + fv["id_vaca"] + "/" + fv["archivo_jpg"].str.split("/").str[-1]

display(fv["carpeta"].value_counts().to_frame("fotos"))
print(f"Marcas en vistas.csv que no corresponden a ninguna foto: {(~vis['archivo_jpg'].isin(f['archivo_jpg'])).sum()}")

**Resultados:** De 333 fotos, 214 son posteriores, 58 laterales izquierdas, 53 laterales derechas y 8 descartadas. Ninguna quedó sin marcar.

### Etiqueta de BCS

In [ ]:
b = bit_i[bit_i["excluida"] == "no"].copy()
b["bcs_num"] = pd.to_numeric(b["bcs"], errors="coerce")
b = b[b["bcs_num"].notna()].copy()
b["fila_num"] = pd.to_numeric(b["fila_excel"], errors="coerce")
b = b.sort_values(["id_vaca", "fila_num"])
b["orden_evaluacion"] = b.groupby("id_vaca").cumcount() + 1
cb = b.groupby("id_vaca").agg(n_eval=("bcs_num", "size"), bcs_entrenamiento=("bcs_num", "mean"),
                              bcs_min=("bcs_num", "min"), bcs_max=("bcs_num", "max"))
cb["bcs_1"] = b[b["orden_evaluacion"] == 1].set_index("id_vaca")["bcs_num"]
cb["bcs_2"] = b[b["orden_evaluacion"] == 2].set_index("id_vaca")["bcs_num"]
cb["dif_evaluadores"] = (cb["bcs_max"] - cb["bcs_min"]).where(cb["n_eval"] >= 2)
cb = cb.drop(columns=["bcs_min", "bcs_max"]).reset_index()

us = fv["usable"] == "si"
fv["_u"] = us.astype(int)
fv["_post"] = (us & (fv["vista"] == "posterior")).astype(int)
fv["_izq"] = (us & (fv["vista"] == "lateral_izq")).astype(int)
fv["_der"] = (us & (fv["vista"] == "lateral_der")).astype(int)
vac = fv.groupby("id_vaca").agg(n_fotos=("ruta", "size"), n_usables=("_u", "sum"), n_posterior=("_post", "sum"),
                                n_lateral_izq=("_izq", "sum"), n_lateral_der=("_der", "sum")).reset_index()
vac = vac.merge(cb, on="id_vaca", how="left")

sin_usables = vac.loc[vac["n_usables"] == 0, "id_vaca"].tolist()
dif = cb["dif_evaluadores"].dropna()
print(f"Vacas: {len(vac)} | sin fotos usables: {sin_usables}")
print(f"Vacas con dos evaluaciones: {len(dif)} | diferencia media {dif.mean():.3f}, máxima {dif.max():.2f}, idénticas {(dif == 0).sum()}")
print(f"Vacas sin ninguna foto lateral usable: {((vac['n_lateral_izq'] == 0) & (vac['n_lateral_der'] == 0) & (vac['n_usables'] > 0)).sum()}")

**Resultados:** Quedan 67 vacas con fotos; 1 no tiene fotos usables (1554). 13 vacas tienen dos evaluaciones, con una diferencia media de 0.115 y máxima de 0.25. 22 vacas no tienen fotos laterales.

### Pliegues de validación cruzada

In [ ]:
eleg = vac[(vac["n_usables"] >= 1) & vac["bcs_entrenamiento"].notna()].copy()
eleg["_idn"] = pd.to_numeric(eleg["id_vaca"], errors="coerce")
eleg = eleg.sort_values(["bcs_entrenamiento", "_idn", "id_vaca"], na_position="last")
eleg["fold"] = np.arange(len(eleg)) % K
vac = vac.merge(eleg[["id_vaca", "fold"]], on="id_vaca", how="left")
vac["fold"] = vac["fold"].astype("Int64")
vac["en_entrenamiento"] = np.where(vac["fold"].notna(), "si", "no")
vac["n_eval"] = vac["n_eval"].astype("Int64")

resumen_folds = (vac[vac["en_entrenamiento"] == "si"].groupby("fold")
                 .agg(vacas=("id_vaca", "size"), fotos=("n_usables", "sum"), posterior=("n_posterior", "sum"),
                      lateral_izq=("n_lateral_izq", "sum"), lateral_der=("n_lateral_der", "sum"),
                      bcs_medio=("bcs_entrenamiento", "mean")).round({"bcs_medio": 2}))
display(resumen_folds)

**Resultados:** Las 66 vacas se repartieron en 5 pliegues de 13 a 14 vacas y de 57 a 76 fotos. El BCS medio por pliegue va de 3.00 a 3.05.

## 3.4 Integrar los datos

In [ ]:
COLS_INDICE = ["ruta", "id_vaca", "vista", "usable", "usar_en_entrenamiento", "fold", "bcs_entrenamiento",
               "bcs_1", "bcs_2", "n_eval", "dif_evaluadores", "archivo_jpg", "archivo_original", "sha256",
               "fecha_hora", "camara", "ancho", "alto", "profundidad", "marcado_por", "marcado_en"]
COLS_VACAS = ["id_vaca", "en_entrenamiento", "fold", "bcs_entrenamiento", "bcs_1", "bcs_2", "n_eval",
              "dif_evaluadores", "n_fotos", "n_usables", "n_posterior", "n_lateral_izq", "n_lateral_der"]
COLS_EVAL = ["fila_excel", "id_vaca", "hora", "bcs", "orden_evaluacion", "n_eval", "fold", "en_entrenamiento"]

datos_vaca = vac[["id_vaca", "fold", "en_entrenamiento", "bcs_entrenamiento", "bcs_1", "bcs_2", "n_eval", "dif_evaluadores"]]
idx = fv.merge(datos_vaca, on="id_vaca", how="left")
idx["usar_en_entrenamiento"] = np.where((idx["usable"] == "si") & (idx["en_entrenamiento"] == "si"), "si", "no")

ev = b.merge(cb[["id_vaca", "n_eval"]], on="id_vaca", how="left")
ev = ev.merge(vac[["id_vaca", "fold", "en_entrenamiento"]], on="id_vaca", how="left")
ev["en_entrenamiento"] = ev["en_entrenamiento"].fillna("no")
ev["bcs"] = ev["bcs_num"]
ev["n_eval"] = ev["n_eval"].astype("Int64")


def por_id(df, extra):
    df = df.assign(_idn=pd.to_numeric(df["id_vaca"], errors="coerce"))
    return df.sort_values(["_idn", "id_vaca"] + extra, na_position="last")


indice = por_id(idx, ["ruta"])[COLS_INDICE]
vacas = por_id(vac, [])[COLS_VACAS]
evaluaciones = ev.sort_values("fila_num")[COLS_EVAL]

usar = indice[indice["usar_en_entrenamiento"] == "si"]
print(f"indice_imagenes: {len(indice)} fotos | para entrenar: {len(usar)} de {usar['id_vaca'].nunique()} vacas")
print(f"vacas: {len(vacas)} | con pliegue: {(vacas['en_entrenamiento'] == 'si').sum()}")
print(f"evaluaciones: {len(evaluaciones)} | de vacas en entrenamiento: {(evaluaciones['en_entrenamiento'] == 'si').sum()}")

**Resultados:** El índice tiene 333 fotos; 325 fotos de 66 vacas se usan para entrenar. Hay 81 evaluaciones, 79 de vacas que entran al entrenamiento.

## 3.5 Formatear los datos (segunda parte): dataset final

In [ ]:
SALIDA.mkdir(exist_ok=True)
copiadas = iguales = 0
for fila in indice.itertuples(index=False):
    src, dst = INTER / fila.archivo_jpg, SALIDA / fila.ruta
    if dst.is_file() and dst.stat().st_size == src.stat().st_size:
        iguales += 1
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copiadas += 1

indice.to_csv(SALIDA / "indice_imagenes.csv", index=False)
vacas.to_csv(SALIDA / "vacas.csv", index=False)
evaluaciones.to_csv(SALIDA / "evaluaciones.csv", index=False)

esperadas = {str(Path(r)) for r in indice["ruta"]}
en_destino = {str(p.relative_to(SALIDA)) for p in (SALIDA / "imagenes").rglob("*") if p.is_file()}
faltan, sobran = esperadas - en_destino, en_destino - esperadas
tam = sum(p.stat().st_size for p in SALIDA.rglob("*") if p.is_file())
print(f"Fotos copiadas ahora: {copiadas} | ya estaban: {iguales} | tamaño del dataset: {tam / 1e9:.2f} GB")
print(f"Rutas del índice que faltan: {len(faltan)} | archivos que sobran: {len(sobran)}")
display(indice["ruta"].str.split("/").str[1].value_counts().to_frame("fotos por carpeta"))

**Resultados:** El dataset final tiene 333 fotos (1.61 GB) en 4 carpetas por vista. Faltan 0 rutas y sobran 0 archivos.

### Verificación contra la referencia

Si al inicio ya había un `indice_imagenes.csv` en `data/dataset_bcs/`, se compara con el nuevo.

In [ ]:
NUMERICAS = {"fold", "bcs_entrenamiento", "bcs_1", "bcs_2", "n_eval", "dif_evaluadores", "ancho", "alto"}
if referencia is None:
    print("No había índice de referencia; no hay nada que comparar.")
else:
    nuevo = leer(SALIDA / "indice_imagenes.csv").sort_values("ruta").reset_index(drop=True)
    ref = referencia.sort_values("ruta").reset_index(drop=True)
    if list(nuevo.columns) != list(ref.columns) or len(nuevo) != len(ref):
        print(f"Distinto: columnas iguales = {list(nuevo.columns) == list(ref.columns)}, filas {len(nuevo)} vs {len(ref)}")
    else:
        difs = []
        for c in nuevo.columns:
            if c in NUMERICAS:
                a, r = pd.to_numeric(nuevo[c], errors="coerce"), pd.to_numeric(ref[c], errors="coerce")
                malas = ~(((a - r).abs() < 1e-9) | (a.isna() & r.isna()))
            else:
                malas = nuevo[c] != ref[c]
            if malas.any():
                difs.append(f"{c}: {int(malas.sum())} diferencias")
        print("El índice coincide con la referencia." if not difs else "Diferencias: " + "; ".join(difs))

**Resultados:** El índice coincide con el del bucket: 333 filas, mismas columnas y ninguna diferencia.

## Comparación con PySpark

La versión con PySpark no está en este notebook. Con los mismos datos, pandas tardó 0.229 s y PySpark 9.237 s (mediana de 5 corridas), y las dos versiones generaron el mismo índice.

## Conclusión

El dataset final tiene 333 fotos de 67 vacas y 81 evaluaciones de BCS. Para entrenar quedan 325 fotos de 66 vacas, repartidas en 5 pliegues por vaca con un BCS medio de entre 3.00 y 3.05. Se excluyeron la vaca 2123 y las 4 fotos compartidas por la 1548 y la 2107; la 2107 y la 1554 no entran al entrenamiento.